In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.8 MB/s eta 0:00:00


In [ ]:
import boto3
import pandas as pd
import json
from io import StringIO
import re
from datetime import datetime

# Función para limpiar microsegundos en las horas
def limpiar_hora(hora_str):
    if '.' in str(hora_str):
        return str(hora_str).split('.')[0]
    else:
        return str(hora_str)

# Función para corregir fechas en formato DD/MM/YY → DD/MM/YYYY
def corregir_fecha(fecha_str):
    try:
        if re.match(r'\d{2}/\d{2}/\d{2}$', str(fecha_str)):
            return datetime.strptime(fecha_str, '%d/%m/%y').strftime('%d/%m/%Y')
        elif re.match(r'\d{2}/\d{2}/\d{4}$', str(fecha_str)):
            return fecha_str
    except:
        pass
    return fecha_str

# Conexión a S3
s3 = boto3.client(
    's3',

#aws_id = os.getenv('AWS_ACCESS_KEY_ID')
#aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

)

bucket_name = 'practica5eli'

# Listar objetos en el bucket
response = s3.list_objects_v2(Bucket=bucket_name)

for obj in response.get('Contents', []):
    archivo_csv = obj['Key']

    if not archivo_csv.endswith('.csv'):
        continue  # Ignorar archivos no CSV

    print(f"Procesando: {archivo_csv}")

    # Descargar CSV desde S3
    csv_obj = s3.get_object(Bucket=bucket_name, Key=archivo_csv)
    body = csv_obj['Body'].read().decode('utf-8')

    # 🔽 Eliminar comillas dobles en todo el contenido
    body = body.replace('"', '')

    # Leer CSV
    df = pd.read_csv(StringIO(body), low_memory=False)

    # Limpiar microsegundos en columnas de hora
    df['Hora_Retiro'] = df['Hora_Retiro'].astype(str).apply(limpiar_hora)
    df['Hora_Arribo'] = df['Hora_Arribo'].astype(str).apply(limpiar_hora)

    # Corregir fechas con años de 2 dígitos
    df['Fecha_Retiro'] = df['Fecha_Retiro'].astype(str).apply(corregir_fecha)
    df['Fecha_Arribo'] = df['Fecha_Arribo'].astype(str).apply(corregir_fecha)

    # Combinar fecha y hora
    df['Inicio_Viaje'] = pd.to_datetime(df['Fecha_Retiro'] + ' ' + df['Hora_Retiro'], dayfirst=True)
    df['Fin_Viaje'] = pd.to_datetime(df['Fecha_Arribo'] + ' ' + df['Hora_Arribo'], dayfirst=True)

    # Seleccionar columnas relevantes y renombrar
    df = df[[
        'Genero_Usuario', 'Edad_Usuario', 'Bici',
        'Ciclo_Estacion_Retiro', 'Ciclo_EstacionArribo',
        'Inicio_Viaje', 'Fin_Viaje'
    ]]
    df.columns = [
        'genero', 'edad', 'bici',
        'estacion_retiro', 'estacion_arribo',
        'inicio_viaje', 'fin_viaje'
    ]

    # Convertir fechas a string para JSON
    df['inicio_viaje'] = df['inicio_viaje'].astype(str)
    df['fin_viaje'] = df['fin_viaje'].astype(str)

    # Convertir a JSON
    json_data = df.to_dict(orient='records')
    json_str = json.dumps(json_data, indent=2, ensure_ascii=False)

    # Guardar JSON en S3 (en carpeta 'procesados')
    nombre_json = archivo_csv.replace('.csv', '.json')
    s3.put_object(Bucket=bucket_name, Key=f'procesados/{nombre_json}', Body=json_str.encode('utf-8'))

    print(f"Guardado: procesados/{nombre_json}")


Procesando: ecobici/2024-08.csv
Guardado: procesados/ecobici/2024-08.json
Procesando: ecobici/2024-09.csv
Guardado: procesados/ecobici/2024-09.json


In [ ]:
pip install pandas pyarrow


In [ ]:
import boto3
import pandas as pd
import json
import random
from io import BytesIO

# Conexión a S3
s3 = boto3.client(
    's3',
#aws_id = os.getenv('AWS_ACCESS_KEY_ID')
#aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

)
bucket_name = 'practica5eli'

# Obtener lista de archivos JSON
response = s3.list_objects_v2(Bucket=bucket_name, Prefix='procesados/')
archivos_json = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.json')]

# Procesar por lotes de 20
tamaño_lote = 2
for i in range(0, len(archivos_json), tamaño_lote):
    lote = archivos_json[i:i + tamaño_lote]
    datos = []

    for archivo in lote:
        print(f"Leyendo: {archivo}")
        contenido = s3.get_object(Bucket=bucket_name, Key=archivo)['Body'].read().decode('utf-8')
        registros = json.loads(contenido)

        # Rellenar valores nulos en Genero_Usuario
        for r in registros:
            if not r.get('genero'):
                r['genero'] = random.choice(['M', 'F'])
        datos.extend(registros)

    # Crear DataFrame
    df_lote = pd.DataFrame(datos)

    # Convertir columnas problemáticas a string
    df_lote['bici'] = df_lote['bici'].astype(str)
    df_lote['edad'] = df_lote['edad'].astype(str)

    # Convertir a Parquet
    buffer = BytesIO()
    df_lote.to_parquet(buffer, index=False)

    # Subir a S3
    nombre_parquet = f'parquets/lote_{i//tamaño_lote + 1}.parquet'
    s3.put_object(Bucket=bucket_name, Key=nombre_parquet, Body=buffer.getvalue())
    print(f"Guardado: {nombre_parquet}")


Leyendo: procesados/ecobici/2010-02-feb.json
Leyendo: procesados/ecobici/2010-03-mar.json
Guardado: parquets/lote_1.parquet
Leyendo: procesados/ecobici/2010-04-abr.json
Leyendo: procesados/ecobici/2010-05-may.json
Guardado: parquets/lote_2.parquet
Leyendo: procesados/ecobici/2010-06-jun.json
Leyendo: procesados/ecobici/2010-07-jul.json
Guardado: parquets/lote_3.parquet
Leyendo: procesados/ecobici/2010-08-ago.json
Leyendo: procesados/ecobici/2010-09-sep.json
Guardado: parquets/lote_4.parquet
Leyendo: procesados/ecobici/2010-10-oct.json
Leyendo: procesados/ecobici/2010-11-nov.json
Guardado: parquets/lote_5.parquet
Leyendo: procesados/ecobici/2010-12-dic.json
Leyendo: procesados/ecobici/2011-01-ene.json
Guardado: parquets/lote_6.parquet
Leyendo: procesados/ecobici/2011-02.json
Leyendo: procesados/ecobici/2011-03.json
Guardado: parquets/lote_7.parquet
Leyendo: procesados/ecobici/2011-04.json
Leyendo: procesados/ecobici/2011-05.json
Guardado: parquets/lote_8.parquet
Leyendo: procesados/ecob